# Continual Pretraining for Sinhala Buddhist Conversational Agent

**Project:** Multi-Lingual Buddhist Conversational Agent  
**Objective:** Continually pretrain a Large Language Model on Sinhala Buddhist corpus

---

## Overview

This notebook performs **continual pretraining** (also called continued pretraining or domain adaptation) on Sinhala Buddhist texts. We will:

1. Load a pretrained base model (Llama 3.1 8B)
2. Extend the tokenizer with Sinhala-specific vocabulary
3. Use **QLoRA** (Quantized Low-Rank Adaptation) for memory-efficient training
4. Train on your `train.txt`, validate on `validation.txt`
5. Save model checkpoints for later instruction fine-tuning

### Why Continual Pretraining?

Base models like Llama are trained primarily on English and high-resource languages. For low-resource languages like Sinhala, and specialized domains like Buddhist texts, continual pretraining helps the model:

- Learn language-specific vocabulary and grammar
- Understand domain-specific terminology (Buddhist concepts)
- Improve subword tokenization efficiency
- Build better contextual representations

---

## Section 1: Environment Setup

### Install Required Libraries

We need several key libraries:
- `transformers`: Hugging Face library for model loading and training
- `peft`: Parameter-Efficient Fine-Tuning (includes LoRA)
- `bitsandbytes`: For 4-bit quantization (QLoRA)
- `accelerate`: For distributed training and mixed precision
- `datasets`: For efficient data loading
- `sentencepiece`: For tokenizer operations
- `tiktoken`: For tokenizer training (optional, if extending vocabulary)


In [10]:
# Upgrade to latest transformers
!pip install --upgrade transformers>=4.43.0

# Also upgrade these for compatibility
!pip install --upgrade accelerate>=0.30.0
!pip install --upgrade bitsandbytes>=0.43.0

# Restart runtime after upgrade
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [1]:
# Install dependencies (run once)
!pip install -q transformers==4.36.0
!pip install -q peft==0.7.1
!pip install -q bitsandbytes==0.41.3
!pip install -q accelerate==0.25.0
!pip install -q datasets==2.15.0
!pip install -q trl==0.7.4  # For training utilities
!pip install -q wandb  # For experiment tracking (optional)
!pip install -q sentencepiece
!pip install -q tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 50.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 17.2 MB

In [1]:
# Import libraries
import os
import torch
import wandb
from pathlib import Path
import json
from datetime import datetime

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)
from datasets import load_dataset, Dataset

# Set random seeds for reproducibility
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.9.0+cu126
CUDA available: True
CUDA version: 12.6
GPU: Tesla T4
GPU Memory: 15.83 GB


---
## Section 2: Configuration

### Define Training Hyperparameters

Based on the SinLlama paper and QLoRA research, here are recommended settings:

In [2]:
# ====================
# CONFIGURATION
# ====================

# Model Configuration
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # Change to "meta-llama/Meta-Llama-3-8B" for base model
# Alternative models:
# "meta-llama/Llama-2-7b-hf"
# "mistralai/Mistral-7B-v0.1"

# Data Paths (MODIFY THESE TO YOUR ACTUAL PATHS)
TRAIN_FILE = "/content/data/sinhala/train.txt"  # Path to your train.txt
VALIDATION_FILE = "/content/data/sinhala/validation.txt"  # Path to your validation.txt
TEST_FILE = "/content/data/sinhala/test.txt"  # Path to your test.txt

# Output Paths
OUTPUT_DIR = "./sinhala_buddhist_model"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"
LOGS_DIR = f"{OUTPUT_DIR}/logs"

# Create directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

# QLoRA Configuration
USE_4BIT = True  # Use 4-bit quantization (QLoRA)
BNB_4BIT_COMPUTE_DTYPE = "bfloat16"  # or "float16"
BNB_4BIT_QUANT_TYPE = "nf4"  # NormalFloat4 (recommended by QLoRA paper)
USE_NESTED_QUANT = True  # Double quantization

# LoRA Configuration (from SinLlama and QLoRA papers)
LORA_R = 64  # Rank - higher = more capacity but more memory
LORA_ALPHA = 16  # Scaling factor (typically r or 2*r)
LORA_DROPOUT = 0.05  # Dropout for smaller models (0.1 for 7B, 0.05 for larger)
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]  # Apply LoRA to all linear layers for best performance

# Training Configuration
MAX_SEQ_LENGTH = 512  # Context length (SinLlama used 512 due to GPU memory)
BATCH_SIZE = 4  # Per device batch size
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
LEARNING_RATE = 2e-4  # Standard for continual pretraining
NUM_EPOCHS = 3  # Adjust based on dataset size
WARMUP_RATIO = 0.03  # 3% warmup steps
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 0.3  # Gradient clipping
LR_SCHEDULER_TYPE = "cosine"  # or "constant"

# Logging and Saving
LOGGING_STEPS = 10
SAVE_STEPS = 500
EVAL_STEPS = 500
SAVE_TOTAL_LIMIT = 3  # Keep only last 3 checkpoints

# WandB (optional - comment out if not using)
USE_WANDB = True
WANDB_PROJECT = "sinhala-buddhist-continual-pretraining"
WANDB_RUN_NAME = f"llama3.1-8b-continual-{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Print configuration
print("=" * 50)
print("TRAINING CONFIGURATION")
print("=" * 50)
print(f"Base Model: {BASE_MODEL}")
print(f"Max Sequence Length: {MAX_SEQ_LENGTH}")
print(f"Batch Size (per device): {BATCH_SIZE}")
print(f"Gradient Accumulation Steps: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective Batch Size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"LoRA Rank: {LORA_R}")
print(f"LoRA Alpha: {LORA_ALPHA}")
print(f"Using 4-bit Quantization: {USE_4BIT}")
print("=" * 50)

TRAINING CONFIGURATION
Base Model: meta-llama/Meta-Llama-3.1-8B-Instruct
Max Sequence Length: 512
Batch Size (per device): 4
Gradient Accumulation Steps: 4
Effective Batch Size: 16
Learning Rate: 0.0002
Epochs: 3
LoRA Rank: 64
LoRA Alpha: 16
Using 4-bit Quantization: True


---
## Section 3: Load and Prepare Data

### Understanding the Data Pipeline

For continual pretraining, we use **Causal Language Modeling (CLM)** objective:
- The model learns to predict the next token given previous tokens
- This is the same objective used in original pretraining
- We chunk text into sequences of `MAX_SEQ_LENGTH` tokens

In [3]:
# Load datasets from text files
print("Loading datasets...")

# Check if files exist
for file_path, name in [(TRAIN_FILE, "Train"), (VALIDATION_FILE, "Validation"), (TEST_FILE, "Test")]:
    if not os.path.exists(file_path):
        print(f"⚠️  Warning: {name} file not found at {file_path}")
    else:
        # Count lines
        with open(file_path, 'r', encoding='utf-8') as f:
            num_lines = sum(1 for _ in f)
        print(f"✓ {name} file found: {num_lines:,} lines")

# Load datasets using Hugging Face datasets
dataset = load_dataset(
    'text',
    data_files={
        'train': TRAIN_FILE,
        'validation': VALIDATION_FILE,
        'test': TEST_FILE
    }
)

print("\nDataset structure:")
print(dataset)

# Display sample
print("\n" + "="*50)
print("SAMPLE FROM TRAINING DATA")
print("="*50)
print(dataset['train'][0]['text'][:500])  # First 500 characters
print("...")

Loading datasets...
✓ Train file found: 372,431 lines
✓ Validation file found: 46,553 lines
✓ Test file found: 46,555 lines


Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]


Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 372431
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 46553
    })
    test: Dataset({
        features: ['text'],
        num_rows: 46555
    })
})

SAMPLE FROM TRAINING DATA
සද්ධර්මරත්නාවලි ය " බොහෝ වහන්දෑ සුප්පබුද්ධ කුෂ්ඨින් මළ නියා ව බුදුන්ට දන්වා ලා ‘ස්වාමීනි, උන් මිය උපන්නේ කොයි ද
...


---
## Section 4: Load Model and Tokenizer

### Understanding QLoRA

**QLoRA** (Quantized Low-Rank Adaptation) combines:
1. **4-bit Quantization**: Reduces model memory from ~16GB to ~5GB for 8B models
2. **LoRA**: Trains small adapter layers instead of full model
3. **Paged Optimizers**: Handles memory spikes during training

This allows training 8B models on a single 24GB GPU!

In [4]:
# Configure quantization
if USE_4BIT:
    print("Configuring 4-bit quantization (QLoRA)...")

    compute_dtype = getattr(torch, BNB_4BIT_COMPUTE_DTYPE)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=USE_NESTED_QUANT,
    )
    print(f"  Quantization Type: {BNB_4BIT_QUANT_TYPE}")
    print(f"  Compute Dtype: {BNB_4BIT_COMPUTE_DTYPE}")
    print(f"  Double Quantization: {USE_NESTED_QUANT}")
else:
    bnb_config = None
    print("Using 16-bit precision (no quantization)")

Configuring 4-bit quantization (QLoRA)...
  Quantization Type: nf4
  Compute Dtype: bfloat16
  Double Quantization: True


In [5]:
# Load tokenizer
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    use_fast=True,  # Use fast tokenizer
)

# Set padding token (needed for batch processing)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"✓ Tokenizer loaded")
print(f"  Vocabulary size: {len(tokenizer):,}")
print(f"  Model max length: {tokenizer.model_max_length}")
print(f"  Padding token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")
print(f"  EOS token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")

# Test tokenization on Sinhala text
print("\n" + "="*50)
print("TESTING SINHALA TOKENIZATION")
print("="*50)
test_text = dataset['train'][0]['text'][:200]  # First 200 chars
tokens = tokenizer.tokenize(test_text)
token_ids = tokenizer.encode(test_text)

print(f"Original text: {test_text}")
print(f"\nNumber of tokens: {len(tokens)}")
print(f"First 10 tokens: {tokens[:10]}")
print(f"\nTokens per character ratio: {len(tokens)/len(test_text):.2f}")
print("  (Lower is better - around 0.5-1.0 is good for Sinhala)")


Loading tokenizer...
✓ Tokenizer loaded
  Vocabulary size: 128,256
  Model max length: 131072
  Padding token: '<|eot_id|>' (ID: 128009)
  EOS token: '<|eot_id|>' (ID: 128009)

TESTING SINHALA TOKENIZATION
Original text: සද්ධර්මරත්නාවලි ය " බොහෝ වහන්දෑ සුප්පබුද්ධ කුෂ්ඨින් මළ නියා ව බුදුන්ට දන්වා ලා ‘ස්වාමීනි, උන් මිය උපන්නේ කොයි ද

Number of tokens: 197
First 10 tokens: ['à·', 'ĥ', 'à¶', '¯', 'à·', 'Ĭ', 'à¶', '°', 'à¶', '»']

Tokens per character ratio: 1.77
  (Lower is better - around 0.5-1.0 is good for Sinhala)


### ⚠️ Important: Tokenizer Analysis

**Check the tokenization efficiency:**
- If tokens-per-character ratio is > 2.0, the tokenizer is inefficient for Sinhala
- You may want to extend the vocabulary (see SinLlama paper)
- For now, we'll proceed with the existing tokenizer
- Vocabulary extension can be done in a separate step if needed

In [6]:
# Load base model
print("\nLoading base model...")
print("This may take a few minutes...")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config if USE_4BIT else None,
    device_map="auto",  # Automatically distribute across GPUs
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if USE_4BIT else torch.float16,
)

print("✓ Model loaded successfully")

# Print model statistics
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Trainable %: {100 * trainable_params / total_params:.2f}%")


Loading base model...
This may take a few minutes...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

✓ Model loaded successfully

Model Statistics:
  Total parameters: 4,540,600,320
  Trainable parameters: 1,050,939,392
  Trainable %: 23.15%


---
## Section 5: Configure LoRA

### Understanding LoRA Parameters

**Key LoRA parameters:**

1. **Rank (r)**: Dimensionality of the low-rank matrices
   - Higher rank = more capacity but more memory
   - Typical values: 8, 16, 32, 64
   - We use 64 based on SinLlama's success

2. **Alpha (α)**: Scaling factor for LoRA updates
   - Controls the magnitude of adaptations
   - Often set to r or 2*r
   - We use 16 (from QLoRA paper)

3. **Target Modules**: Which layers to apply LoRA to
   - Research shows applying to ALL layers gives best results
   - We target: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj

4. **Dropout**: Regularization
   - 0.05 for models up to 13B
   - Helps prevent overfitting

In [7]:
# Prepare model for k-bit training
if USE_4BIT:
    print("Preparing model for k-bit training...")
    model = prepare_model_for_kbit_training(model)
    print("✓ Model prepared for QLoRA")

# Configure LoRA
print("\nConfiguring LoRA...")
peft_config = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",  # Don't train bias terms
    task_type=TaskType.CAUSAL_LM,  # Causal language modeling
    target_modules=LORA_TARGET_MODULES,
)

# Apply LoRA to model
model = get_peft_model(model, peft_config)

print("✓ LoRA configured and applied")

# Print trainable parameters after LoRA
model.print_trainable_parameters()

# Calculate memory usage
if torch.cuda.is_available():
    memory_allocated = torch.cuda.memory_allocated() / 1e9
    memory_reserved = torch.cuda.memory_reserved() / 1e9
    print(f"\nGPU Memory:")
    print(f"  Allocated: {memory_allocated:.2f} GB")
    print(f"  Reserved: {memory_reserved:.2f} GB")

Preparing model for k-bit training...
✓ Model prepared for QLoRA

Configuring LoRA...
✓ LoRA configured and applied
trainable params: 167,772,160 || all params: 8,198,033,408 || trainable%: 2.0464927580836716

GPU Memory:
  Allocated: 8.48 GB
  Reserved: 11.85 GB


### 📊 Expected Results

You should see:
- **Trainable parameters**: ~0.2-1% of total (much smaller than 8B!)
- **GPU memory**: ~5-7GB for 8B model with 4-bit quantization
- This is the magic of QLoRA! 🎉

---
## Section 6: Prepare Dataset for Training

### Tokenization Strategy

For continual pretraining:
1. Tokenize all text
2. Concatenate into long sequences
3. Chunk into fixed-length blocks (MAX_SEQ_LENGTH)
4. Use Causal Language Modeling objective

In [8]:
# Tokenization function
def tokenize_function(examples):
    """
    Tokenize text examples.
    For continual pretraining, we concatenate all texts and chunk them.
    """
    # Remove empty lines
    texts = [text.strip() for text in examples['text'] if text.strip()]

    # Tokenize
    output = tokenizer(
        texts,
        truncation=False,  # Don't truncate yet
        padding=False,  # Don't pad yet
        return_attention_mask=False,  # We'll handle attention mask later
    )

    return output

# Tokenize datasets
print("Tokenizing datasets...")
print("This may take several minutes depending on dataset size...")

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
    num_proc=4,  # Use multiple processes for speed
    desc="Tokenizing",
)

print("✓ Tokenization complete")
print(tokenized_datasets)

Tokenizing datasets...
This may take several minutes depending on dataset size...


Tokenizing (num_proc=4):   0%|          | 0/372431 [00:00<?, ? examples/s]

Tokenizing (num_proc=4):   0%|          | 0/46553 [00:00<?, ? examples/s]

Tokenizing (num_proc=4):   0%|          | 0/46555 [00:00<?, ? examples/s]

✓ Tokenization complete
DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 372431
    })
    validation: Dataset({
        features: ['input_ids'],
        num_rows: 46553
    })
    test: Dataset({
        features: ['input_ids'],
        num_rows: 46555
    })
})


In [9]:
# Group texts into chunks of MAX_SEQ_LENGTH
def group_texts(examples):
    """
    Concatenate all texts and chunk into blocks of MAX_SEQ_LENGTH.
    This is the standard approach for continual pretraining.
    """
    # Concatenate all texts
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])

    # We drop the last chunk if it's smaller than MAX_SEQ_LENGTH
    total_length = (total_length // MAX_SEQ_LENGTH) * MAX_SEQ_LENGTH

    # Split by chunks of MAX_SEQ_LENGTH
    result = {
        k: [t[i : i + MAX_SEQ_LENGTH] for i in range(0, total_length, MAX_SEQ_LENGTH)]
        for k, t in concatenated_examples.items()
    }

    # Create labels (same as input_ids for causal LM)
    result["labels"] = result["input_ids"].copy()

    return result

print("\nGrouping texts into chunks...")
chunked_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    num_proc=4,
    desc="Grouping texts",
)

print("✓ Text grouping complete")
print("\nFinal dataset sizes:")
print(f"  Training: {len(chunked_datasets['train']):,} examples")
print(f"  Validation: {len(chunked_datasets['validation']):,} examples")
print(f"  Test: {len(chunked_datasets['test']):,} examples")

# Calculate approximate training steps
num_training_steps = (len(chunked_datasets['train']) * NUM_EPOCHS) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print(f"\nApproximate training steps: {num_training_steps:,}")


Grouping texts into chunks...


Grouping texts (num_proc=4):   0%|          | 0/372431 [00:00<?, ? examples/s]

Grouping texts (num_proc=4):   0%|          | 0/46553 [00:00<?, ? examples/s]

Grouping texts (num_proc=4):   0%|          | 0/46555 [00:00<?, ? examples/s]

✓ Text grouping complete

Final dataset sizes:
  Training: 91,814 examples
  Validation: 11,529 examples
  Test: 11,482 examples

Approximate training steps: 17,215


---
## Section 7: Configure Training

### Training Strategy

We use **Causal Language Modeling** with these optimizations:
- **Gradient Accumulation**: Simulate larger batch sizes
- **Mixed Precision (BF16)**: Faster training, lower memory
- **Gradient Checkpointing**: Trade compute for memory
- **Cosine Learning Rate Schedule**: Better convergence
- **Weight Decay**: Regularization

In [10]:
# Initialize WandB (optional)
if USE_WANDB:
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        config={
            "base_model": BASE_MODEL,
            "max_seq_length": MAX_SEQ_LENGTH,
            "batch_size": BATCH_SIZE,
            "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
            "learning_rate": LEARNING_RATE,
            "num_epochs": NUM_EPOCHS,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
        }
    )
    print("✓ WandB initialized")
    print(f"  Project: {WANDB_PROJECT}")
    print(f"  Run: {WANDB_RUN_NAME}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ranidu-h-gurusinghe (ranidu-h-gurusinghe-robert-gordon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ WandB initialized
  Project: sinhala-buddhist-continual-pretraining
  Run: llama3.1-8b-continual-20260120_075522


In [15]:
# Training arguments
training_args = TrainingArguments(
    # Output and logging
    output_dir=CHECKPOINT_DIR,
    logging_dir=LOGS_DIR,
    logging_steps=LOGGING_STEPS,

    # Training hyperparameters
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # Optimizer
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,

    # Learning rate schedule
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_ratio=WARMUP_RATIO,

    # Precision and optimization
    bf16=True if BNB_4BIT_COMPUTE_DTYPE == "bfloat16" else False,
    fp16=True if BNB_4BIT_COMPUTE_DTYPE == "float16" else False,
    gradient_checkpointing=True,  # Trade compute for memory

    # Evaluation and saving
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # WandB
    report_to="wandb" if USE_WANDB else "none",

    # Other
    group_by_length=True,  # Group sequences of similar length for efficiency
    dataloader_num_workers=4,
    remove_unused_columns=False,
)

print("✓ Training arguments configured")
print(f"\nEffective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Total optimization steps: ~{num_training_steps:,}")
print(f"Warmup steps: ~{int(num_training_steps * WARMUP_RATIO):,}")

✓ Training arguments configured

Effective batch size: 16
Total optimization steps: ~17,215
Warmup steps: ~516


In [16]:
# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # We're doing causal LM, not masked LM
)

print("✓ Data collator configured")

✓ Data collator configured


---
## Section 8: Initialize Trainer and Start Training

### What Happens During Training?

1. **Forward Pass**: Model predicts next tokens
2. **Loss Calculation**: Compare predictions to actual next tokens
3. **Backward Pass**: Compute gradients
4. **Optimizer Step**: Update LoRA weights (not base model!)
5. **Evaluation**: Periodically check validation loss
6. **Checkpointing**: Save best models

In [17]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=chunked_datasets['train'],
    eval_dataset=chunked_datasets['validation'],
    data_collator=data_collator,
)

print("✓ Trainer initialized")
print("\nReady to start training!")

✓ Trainer initialized

Ready to start training!


In [18]:
# Train the model
print("\n" + "="*50)
print("STARTING TRAINING")
print("="*50)
print("This will take several hours...")
print("Monitor progress in WandB or in the logs directory.")
print("\nYou can safely interrupt training and resume from checkpoints.")
print("="*50 + "\n")

# Start training
train_result = trainer.train()

# Print training summary
print("\n" + "="*50)
print("TRAINING COMPLETE!")
print("="*50)
print(f"Total training time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Training loss: {train_result.metrics['train_loss']:.4f}")
print(f"Samples per second: {train_result.metrics['train_samples_per_second']:.2f}")
print(f"Steps per second: {train_result.metrics['train_steps_per_second']:.4f}")


STARTING TRAINING
This will take several hours...
Monitor progress in WandB or in the logs directory.

You can safely interrupt training and resume from checkpoints.



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 1002.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 588.12 MiB is free. Process 45777 has 14.16 GiB memory in use. Of the allocated memory 13.65 GiB is allocated by PyTorch, and 396.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

---
## Section 9: Save the Model

### Two Saving Options:

1. **Save LoRA adapters only** (recommended for checkpointing)
   - Small files (~100-500 MB)
   - Need to load base model + adapters for inference

2. **Merge and save full model** (for deployment)
   - Large files (~16 GB for 8B model)
   - Self-contained, ready to use

In [ ]:
# Save LoRA adapters
adapter_path = f"{OUTPUT_DIR}/lora_adapters"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"✓ LoRA adapters saved to: {adapter_path}")
print(f"  These can be loaded with: PeftModel.from_pretrained()")

# Save training configuration
config_path = f"{OUTPUT_DIR}/training_config.json"
config_dict = {
    "base_model": BASE_MODEL,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
    "training_date": datetime.now().isoformat(),
}

with open(config_path, 'w') as f:
    json.dump(config_dict, f, indent=2)

print(f"✓ Training configuration saved to: {config_path}")

In [ ]:
# Optional: Merge LoRA weights into base model and save
# WARNING: This creates a large file (~16GB for 8B model)
# Uncomment if you want a standalone model

SAVE_MERGED_MODEL = False  # Set to True if you want merged model

if SAVE_MERGED_MODEL:
    print("\nMerging LoRA weights with base model...")
    print("This may take a few minutes...")

    # Merge LoRA weights
    model = model.merge_and_unload()

    # Save merged model
    merged_path = f"{OUTPUT_DIR}/merged_model"
    model.save_pretrained(merged_path)
    tokenizer.save_pretrained(merged_path)

    print(f"✓ Merged model saved to: {merged_path}")
    print(f"  This is a standalone model ready for deployment")
else:
    print("\nSkipping merged model save (SAVE_MERGED_MODEL=False)")
    print("To save merged model, set SAVE_MERGED_MODEL=True and re-run this cell")

---
## Section 10: Evaluation

### Evaluation Metrics

For continual pretraining, we primarily track:
1. **Perplexity**: exp(loss) - lower is better
2. **Validation Loss**: How well the model predicts held-out data

In [ ]:
# Evaluate on test set
print("\nEvaluating on test set...")
test_results = trainer.evaluate(eval_dataset=chunked_datasets['test'])

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"Test Loss: {test_results['eval_loss']:.4f}")
print(f"Test Perplexity: {np.exp(test_results['eval_loss']):.2f}")
print("="*50)

# Save test results
test_results_path = f"{OUTPUT_DIR}/test_results.json"
with open(test_results_path, 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"\n✓ Test results saved to: {test_results_path}")

---
## Section 11: Test Generation (Qualitative Check)

Let's test if the model can generate Sinhala text!

In [ ]:
# Generate some sample text
print("\nTesting text generation...")
print("="*50)

# Get a sample prompt from training data
sample_prompt = dataset['test'][0]['text'][:100]  # First 100 chars as prompt

print(f"Prompt: {sample_prompt}")
print("\nGenerating...\n")

# Tokenize prompt
inputs = tokenizer(sample_prompt, return_tensors="pt").to(model.device)

# Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode and print
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Generated: {generated_text}")
print("="*50)

print("\n✓ Generation test complete")
print("\nNote: The quality will improve with instruction fine-tuning!")

---
## Section 12: Summary and Next Steps

### 🎉 Congratulations!

You've successfully performed continual pretraining on your Sinhala Buddhist corpus!

### What You've Accomplished:

1. ✓ Loaded and prepared Sinhala Buddhist text corpus
2. ✓ Configured QLoRA for memory-efficient training
3. ✓ Applied LoRA adapters to base model
4. ✓ Trained the model on domain-specific data
5. ✓ Evaluated performance on test set
6. ✓ Saved model checkpoints

### 📂 Saved Files:

```
sinhala_buddhist_model/
├── lora_adapters/          # LoRA weights (load these!)
├── checkpoints/            # Training checkpoints
├── logs/                   # Training logs
├── training_config.json    # Configuration
└── test_results.json       # Evaluation results
```

### 🚀 Next Steps:

1. **Generate Q&A Pairs**: Use LLMs to create instruction data
2. **Instruction Fine-tuning**: Train on Q&A pairs
3. **RAG Integration**: Build retrieval system
4. **Evaluation**: Test on Buddhist Q&A benchmarks
5. **Deployment**: Create web interface

### 📊 How to Load Your Model:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)

# Load LoRA adapters
model = PeftModel.from_pretrained(base_model, "./sinhala_buddhist_model/lora_adapters")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("./sinhala_buddhist_model/lora_adapters")
```

### ❓ Questions or Issues?

- Check WandB for training curves and metrics
- Review logs in `./sinhala_buddhist_model/logs/`
- Compare perplexity: Lower is better!
- Test generation with different prompts

---

In [19]:
# Close WandB
if USE_WANDB:
    wandb.finish()
    print("✓ WandB run completed")

print("\n" + "="*50)
print("ALL DONE! 🎉")
print("="*50)
print(f"\nYour model is saved in: {OUTPUT_DIR}")
print("\nNext: Create instruction dataset and fine-tune for Q&A!")

✓ WandB run completed

ALL DONE! 🎉

Your model is saved in: ./sinhala_buddhist_model

Next: Create instruction dataset and fine-tune for Q&A!
